In [ ]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
import reverse_geocoder as rg # type: ignore
# -------------------------------------------------
from sklearn.linear_model import LinearRegression

In [12]:
data = pd.read_excel("Crime_Incidents_in_the_Last_30_Days.xlsx")
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [5]:
data.head(5)
# data.info()
# data.describe()
# data.groupby(['NEIGHBORHOOD_CLUSTER','PSA','DISTRICT','ANC','WARD'])['BID'].value_counts()

,CCN,SHIFT,METHOD,OFFENSE,BLOCK,ANC,DISTRICT,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP-CENSUS_TRACT & WARD,VOTING_PRECINCT,LATITUDE,LONGITUDE,BID,START_DATE,END_DATE,REPORT_DAT,INCIDENT_DURATION,REPORTING_GAP,AREA_NAME
0,25161987,MIDNIGHT,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,1A,3,302,Cluster 2,2802 1,Precinct 39,38.929517,-77.032730,None,2025-10-25 07:55:00,2025-10-25 01:55:00,2025-10-27 06:00:00,6H,52H 5M,"Washington, D.C."
1,25162437,EVENING,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,1A,3,302,Cluster 2,2802 1,Precinct 39,38.929517,-77.032730,None,2025-10-18 06:00:00,2025-10-18 03:07:00,2025-10-19 02:52:00,2H 53M,23H 45M,"Washington, D.C."
2,25157565,DAY,GUN,ROBBERY,1400 - 1599 BLOCK OF NEWTON STREET NW,1A,4,408,Cluster 2,2801 1,Precinct 41,38.932567,-77.034601,None,2025-10-18 21:00:00,2025-10-18 12:03:00,2025-10-23 08:56:00,8H 57M,116H 53M,"Washington, D.C."
3,25160412,EVENING,OTHERS,THEFT/OTHER,2818 - 2899 BLOCK OF SHERMAN AVENUE NW,1A,3,304,Cluster 2,3500 1,Precinct 37,38.926607,-77.025900,None,2025-10-21 22:19:00,2025-10-21 23:04:00,2025-10-22 03:05:00,45M,4H 1M,"Washington, D.C."
4,25163492,DAY,OTHERS,THEFT/OTHER,3100 - 3299 BLOCK OF 14TH STREET NW,1A,3,302,Cluster 2,2802 1,Precinct 39,38.929517,-77.032730,None,2025-10-27 18:32:00,2025-10-27 19:30:00,2025-10-27 07:43:00,58M,11H 47M,"Washington, D.C."


In [14]:

# fig1 = px.box(data,x ='METHOD',y='OFFENSE',color='OFFENSE',template='plotly_dark')
# fig1.show()

# fig2 = px.area(data.groupby(['OFFENSE','NEIGHBORHOOD_CLUSTER','BLOCK_GROUP-CENSUS_TRACT & WARD','AREA_NAME'])['DISTRICT'],template='plotly_dark')
# fig2.show()

fig3 = px.box(data,x='NEIGHBORHOOD_CLUSTER',y='AREA_NAME',template='plotly_dark')
fig3.show()

# fig4 = px.pie(data,names='OFFENSE',values=data['INCIDENT_DURATION'].apply(convert_to_hours_as_nums),template='plotly_dark')
# fig4.show()

# fig5 = px.treemap(data,path=['OFFENSE','METHOD','SHIFT'],template='plotly_dark')
# fig5.show()

# fig6 = px.area(data, x='INCIDENT_DURATION', y='REPORTING_GAP',template='plotly_dark')
# fig6.show()




In [13]:
data['DISTRICT'] = data['DISTRICT'].convert_dtypes(int)
data['PSA'] = data['PSA'].convert_dtypes(int)
data = data.drop(columns = ['XBLOCK','YBLOCK','CENSUS_TRACT','WARD'])
data = data.rename(columns = {'BLOCK_GROUP':'BLOCK_GROUP-CENSUS_TRACT & WARD'})
data['BLOCK_GROUP-CENSUS_TRACT & WARD'] = data['BLOCK_GROUP-CENSUS_TRACT & WARD'].astype(str).str.lstrip('0')
data = data.where(data.notna(), None)
  
data['START_DATE'] = pd.to_datetime(data['START_DATE'])
data['END_DATE'] = pd.to_datetime(data['END_DATE'])
data['REPORT_DAT'] = pd.to_datetime(data['REPORT_DAT'])

data['NEIGHBORHOOD_CLUSTER'] = pd.Categorical(data['NEIGHBORHOOD_CLUSTER'],categories=sorted(data['NEIGHBORHOOD_CLUSTER'].dropna().unique(), key=lambda x: int(re.search(r'\d+', x).group())),ordered=True)
data = data.sort_values('NEIGHBORHOOD_CLUSTER')
  
data['incident_duration'] = ((data['END_DATE'] - data['START_DATE']).dt.total_seconds() / 60).abs()
data['reporting_gap'] = ((data['REPORT_DAT'] - data['END_DATE']).dt.total_seconds() / 60).abs()
  
coords = list(zip(data['LATITUDE'], data['LONGITUDE']))   #type: ignore
results = rg.search(coords)
data['area_name'] = [r['name'] for r in results]   #type: ignore
  
data.columns = data.columns.str.upper()

In [4]:
  
def format_duration(mins) :
    if mins < 0 :
        return None
    hours = int(mins // 60)
    remaining_mins = int(mins % 60)
    if hours == 0 :
        return f"{remaining_mins}M"
    elif remaining_mins == 0 :
        return f"{hours}H"
    else :
        return f"{hours}H {remaining_mins}M"
    
data['INCIDENT_DURATION'] = data ['INCIDENT_DURATION'].apply(format_duration)
  
def format_duration(mins) :
    if mins < 0 :
        return None
    hours = int(mins // 60)
    remaining_mins = int(mins % 60)
    if hours == 0 :
        return f"{remaining_mins}M"
    elif remaining_mins == 0 :
        return f"{hours}H"
    else :
        return f"{hours}H {remaining_mins}M"
    
data['REPORTING_GAP'] = data ['REPORTING_GAP'].apply(format_duration)
  
def convert_to_hours_as_nums(x) :
    h = 0
    m = 0
    
    if 'H' in x :
        h = int(x.split('H')[0])
        x = x.split('H')[1]
        
    if 'M' in x :
        m = int(x.replace('M',""))
        
    return h + m/60

In [12]:
data.to_excel('crime_data.xlsx', index=False)

In [36]:
data['SHIFT'].value_counts()

In [37]:
data['METHOD'].value_counts()

In [38]:
data['OFFENSE'].value_counts()

In [39]:
data.isnull().sum()